# 02 · Statistical Analysis (Q1)

**คำถามวิจัย Q1** — ปัจจัยใดสัมพันธ์กับ Stroke อย่างมีนัยสำคัญ?

เลือกการทดสอบตาม **ชนิดของตัวแปร** ไม่ใช่ใช้ t-test กับทุกอย่าง:

| ตัวแปร | การทดสอบ | Effect size |
|---|---|---|
| เชิงกลุ่ม / ไบนารี | Chi-square test of independence | Cramér's V |
| ต่อเนื่อง | Mann-Whitney U (ไม่ต้องสมมติ normal) | rank-biserial correlation |
| ไบนารี (ความเสี่ยง) | Odds Ratio + 95% CI | — |

**ปรับ p-value ด้วย Benjamini-Hochberg (FDR)** เพราะทดสอบพร้อมกันหลายตัวแปร
→ ลดปัญหา false positive inflation

## 1. เตรียมข้อมูล

In [1]:
import sys
from pathlib import Path

# ให้ import src ได้ไม่ว่าจะเปิด notebook จากโฟลเดอร์ไหน
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
print("Project root:", ROOT)

Project root: D:\Stroke Prediction Dataset\Stroke-Prediction-XAI


In [2]:
from src import config as cfg
from src import data_loader as dl
from src import stats_analysis as sa

clean = dl.basic_clean(dl.load_raw())
print("n =", len(clean), "| stroke =", int(clean[cfg.TARGET].sum()))

n = 5109 | stroke = 249


## 2. ตัวแปรเชิงกลุ่ม vs Stroke — Chi-square

**สมมติฐาน**
- H₀ : สัดส่วนการเกิด stroke ไม่ขึ้นกับตัวแปรนั้น (independent)
- H₁ : มีความสัมพันธ์กัน

เกณฑ์ตัดสิน: ใช้ **p-value ที่ปรับ FDR แล้ว** < 0.05 **ร่วมกับ** ขนาด effect size
(เพราะข้อมูล 5,000 แถวทำให้ p ต่ำได้ง่ายแม้ความสัมพันธ์จะเล็กมาก)

In [3]:
chi = sa.chi_square_vs_target(clean)
chi[["feature", "chi2", "dof", "p_value", "p_adjusted_bh",
     "cramers_v", "effect_size", "significant_0.05", "warning"]]

,feature,chi2,dof,p_value,p_adjusted_bh,cramers_v,effect_size,significant_0.05,warning
0,gender,0.3400,1,5.598278e-01,5.598278e-01,0.0000,น้อยมาก (negligible),False,
1,ever_married,58.8678,1,1.686286e-14,3.934666e-14,0.1074,น้อย (small),True,
2,work_type,49.1591,4,5.409035e-10,9.465812e-10,0.0940,น้อยมาก (negligible),True,
3,Residence_type,1.0750,1,2.998252e-01,3.497961e-01,0.0065,น้อยมาก (negligible),False,
4,smoking_status,29.2257,3,2.007704e-06,2.810786e-06,0.0717,น้อยมาก (negligible),True,
5,hypertension,81.5731,1,1.688936e-19,5.911277e-19,0.1271,น้อย (small),True,
6,heart_disease,90.2294,1,2.120831e-21,1.484582e-20,0.1342,น้อย (small),True,


In [4]:
(chi.assign(stroke_rate=chi["stroke_rate_by_group_%"])
    [["feature", "p_adjusted_bh", "cramers_v", "stroke_rate"]]
    .sort_values("cramers_v", ascending=False)
    .style.background_gradient(subset=["cramers_v"], cmap="Reds")
    .format({"p_adjusted_bh": "{:.2e}"}))

,feature,p_adjusted_bh,cramers_v,stroke_rate
6,heart_disease,1.48e-20,0.134200,0: 4.18 | 1: 17.03
5,hypertension,5.91e-19,0.127100,0: 3.97 | 1: 13.25
1,ever_married,3.93e-14,0.107400,No: 1.65 | Yes: 6.56
2,work_type,9.47e-10,0.094000,Govt_job: 5.02 | Never_worked: 0.0 | Private: 5.1 | Self-employed: 7.94 | children: 0.29
4,smoking_status,2.81e-06,0.071700,Unknown: 3.04 | formerly smoked: 7.92 | never smoked: 4.76 | smokes: 5.32
3,Residence_type,3.50e-01,0.006500,Rural: 4.54 | Urban: 5.2
0,gender,5.60e-01,0.000000,Female: 4.71 | Male: 5.11


**เกณฑ์ตีความ Cramér's V** (Cohen): < 0.10 น้อยมาก | 0.10–0.19 น้อย | 0.20–0.29 ปานกลาง | ≥ 0.30 มาก

> **Insight Q1.1** — `heart_disease` (V = 0.134) และ `hypertension` (V = 0.127)
> มีความสัมพันธ์แรงที่สุดในกลุ่มตัวแปรเชิงกลุ่ม แต่อยู่ในระดับ "น้อย"
> ส่วน `gender` และ `Residence_type` **ไม่มีนัยสำคัญ** (p > 0.05)

## 3. ตัวแปรต่อเนื่อง vs Stroke — Mann-Whitney U

**ทำไมไม่ใช้ t-test?** เพราะ t-test สมมติว่าข้อมูลเป็น normal distribution
เราจึงตรวจ normality ด้วย **Shapiro-Wilk** ก่อน (คอลัมน์ `shapiro_p_*`)
ถ้า p < 0.05 = ปฏิเสธ normality → ใช้ Mann-Whitney U ซึ่งเป็น non-parametric

In [5]:
mw = sa.mann_whitney_vs_target(clean)
mw[["feature", "median_no_stroke", "median_stroke", "mean_no_stroke", "mean_stroke",
    "u_statistic", "p_value", "p_adjusted_bh", "rank_biserial",
    "shapiro_p_no_stroke", "shapiro_p_stroke", "normality_rejected", "significant_0.05"]]

,feature,median_no_stroke,median_stroke,mean_no_stroke,mean_stroke,u_statistic,p_value,p_adjusted_bh,rank_biserial,shapiro_p_no_stroke,shapiro_p_stroke,normality_rejected,significant_0.05
0,age,43.00,71.00,41.97,67.73,1009878.5,3.849734e-71,1.154920e-70,0.6690,0.0,0.000000,True,True
1,avg_glucose_level,91.47,105.22,104.79,132.54,739058.0,3.583095e-09,5.374642e-09,0.2214,0.0,0.000000,True,True
2,bmi,28.00,29.70,28.82,30.47,568828.0,1.041877e-04,1.041877e-04,0.1584,0.0,0.000005,True,True


In [6]:
mw[["feature", "rank_biserial", "direction"]].style \
    .background_gradient(subset=["rank_biserial"], cmap="RdBu_r", vmin=-1, vmax=1)

,feature,rank_biserial,direction
0,age,0.669000,กลุ่ม stroke มีค่าสูงกว่า
1,avg_glucose_level,0.221400,กลุ่ม stroke มีค่าสูงกว่า
2,bmi,0.158400,กลุ่ม stroke มีค่าสูงกว่า


**อ่านค่า rank-biserial**: ค่าอยู่ในช่วง −1 ถึง +1
- ค่าบวก = กลุ่ม stroke มีค่าสูงกว่ากลุ่มไม่เป็น stroke
- |ค่า| ยิ่งมาก = ทั้งสองกลุ่มยิ่งแยกออกจากกันชัด

> **Insight Q1.2** — `age` มีค่า rank-biserial = **0.669** ซึ่งสูงมาก
> (มัธยฐาน 71 ปี vs 43 ปี) นี่คือหลักฐานเชิงสถิติที่แข็งแรงที่สุดในชุดข้อมูลนี้
> ขณะที่ `avg_glucose_level` (0.221) และ `bmi` (0.158) มีนัยสำคัญแต่ขนาดผลเล็กกว่ามาก

## 4. Odds Ratio — ขนาดความเสี่ยง

In [7]:
or_df = sa.odds_ratios(clean)
or_df

,feature,odds_ratio,ci_95_low,ci_95_high,stroke_rate_exposed_%,stroke_rate_unexposed_%,interpretation
0,hypertension,3.697,2.743,4.981,13.25,3.97,เพิ่มความเสี่ยง
1,heart_disease,4.705,3.336,6.637,17.03,4.18,เพิ่มความเสี่ยง


**ตีความ**: OR = 4.71 ของ `heart_disease` หมายความว่า
ผู้ที่มีโรคหัวใจมี **odds** ของการเกิด stroke สูงเป็น **4.71 เท่า** ของผู้ที่ไม่มี

> **Insight Q1.3** — ปัจจัยเสี่ยงที่ "แก้ไขไม่ได้" (อายุ, โรคหัวใจ, ความดัน)
> มีพลังในการทำนายสูงกว่าปัจจัยด้านพฤติกรรมอย่างการสูบบุหรี่/สถานะสมรส
> ซึ่งอธิบายได้ว่าทำไมโมเดลในเล่ม 03 จึงให้ความสำคัญกับอายุเป็นอันดับหนึ่ง

## 5. สรุป Q1 (สรุปเป็นตารางเดียว)

In [8]:
rows = []
for _, r in chi.iterrows():
    rows.append({"feature": r["feature"], "type": "categorical", "test": "Chi-square",
                 "p_adjusted": r["p_adjusted_bh"], "effect_size": r["cramers_v"],
                 "metric": "Cramér's V", "significant": r["significant_0.05"]})
for _, r in mw.iterrows():
    rows.append({"feature": r["feature"], "type": "continuous", "test": "Mann-Whitney U",
                 "p_adjusted": r["p_adjusted_bh"], "effect_size": abs(r["rank_biserial"]),
                 "metric": "rank-biserial", "significant": r["significant_0.05"]})

final = pd.DataFrame(rows).sort_values(["significant", "effect_size"], ascending=False)
final.style.background_gradient(subset=["effect_size"], cmap="Blues").format({"p_adjusted": "{:.2e}"})

,feature,type,test,p_adjusted,effect_size,metric,significant
7,age,continuous,Mann-Whitney U,1.15e-70,0.669000,rank-biserial,True
8,avg_glucose_level,continuous,Mann-Whitney U,5.37e-09,0.221400,rank-biserial,True
9,bmi,continuous,Mann-Whitney U,1.04e-04,0.158400,rank-biserial,True
6,heart_disease,categorical,Chi-square,1.48e-20,0.134200,Cramér's V,True
5,hypertension,categorical,Chi-square,5.91e-19,0.127100,Cramér's V,True
1,ever_married,categorical,Chi-square,3.93e-14,0.107400,Cramér's V,True
2,work_type,categorical,Chi-square,9.47e-10,0.094000,Cramér's V,True
4,smoking_status,categorical,Chi-square,2.81e-06,0.071700,Cramér's V,True
3,Residence_type,categorical,Chi-square,3.50e-01,0.006500,Cramér's V,False
0,gender,categorical,Chi-square,5.60e-01,0.000000,Cramér's V,False


In [9]:
final.to_csv(cfg.TABLES_DIR / "q1_summary_combined.csv", index=False, encoding="utf-8-sig")
print("บันทึกตารางสรุปแล้ว")

บันทึกตารางสรุปแล้ว


## 6. คำตอบของ Q1

ปัจจัยที่มีความสัมพันธ์กับ stroke อย่างมีนัยสำคัญทางสถิติ (FDR < 0.05) เรียงตามขนาดผล:

| อันดับ | ปัจจัย | ขนาดผล | ทิศทาง |
|---|---|---|---|
| 1 | **age** | rank-biserial 0.669 | ยิ่งอายุมาก ยิ่งเสี่ยง |
| 2 | **avg_glucose_level** | rank-biserial 0.221 | น้ำตาลสูง → เสี่ยงขึ้น |
| 3 | **bmi** | rank-biserial 0.158 | BMI สูง → เสี่ยงขึ้นเล็กน้อย |
| 4 | **heart_disease** | Cramér's V 0.134, OR 4.71 | มีโรคหัวใจ → เสี่ยงสูงมาก |
| 5 | **hypertension** | Cramér's V 0.127, OR 3.70 | มีความดันสูง → เสี่ยงสูง |
| 6 | **ever_married** | Cramér's V 0.107 | สมรสแล้ว → เสี่ยงขึ้น (likely confounded by age) |
| 7 | **work_type** | Cramér's V 0.094 | ต่างกันตามกลุ่มอาชีพ |
| 8 | **smoking_status** | Cramér's V 0.072 | มีนัยสำคัญแต่ผลเล็ก (และมี Unknown 30%) |

**ไม่มีนัยสำคัญ**: `gender` (p = 0.56), `Residence_type` (p = 0.30)

> ⚠️ **ข้อควรระวัง**: `ever_married` และ `work_type` น่าจะเป็น *proxy* ของอายุ
> (คนอายุมากย่อมมีโอกาสสมรสและเกษียณมากกว่า) จึงควรตีความว่าเป็น
> **ความสัมพันธ์ (association)** ไม่ใช่ **สาเหตุ (causation)**
> และควรตรวจ confounding เพิ่มเติมด้วย multivariable model